# Silver Transactions

Convert released Bronze transactions into trusted monitoring inputs.

Input: workspace.sentinel_bronze.transactions

Planned outputs:
- Accepted transactions.
- Quarantined records with validation reasons.

Begin by inspecting schema and amount precision.
No tables are written during this inspection.

In [ ]:
from pyspark.sql import functions as F

# Read the persisted Bronze table, not future staged batches.
bronze_table = "workspace.sentinel_bronze.transactions"
bronze = spark.table(bronze_table)

# Inspect actual types before defining Silver transformations.
bronze.printSchema()

# Labels must remain outside operational data.
assert "is_fraud" not in bronze.columns, (
    "Unexpected fraud labels in Bronze"
)

# Confirm which released batches are available.
display(
    bronze
    .groupBy("replay_batch")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy("replay_batch")
)

# Compare a wider decimal representation with the intended monetary type.
# A successful DECIMAL(18,2) cast alone does not prove that no rounding occurred.
amount_precision = bronze.select(
    F.expr("TRY_CAST(amt AS DECIMAL(38, 18))").alias("wide_amount"),
    F.expr("TRY_CAST(amt AS DECIMAL(18, 2))").alias("money_amount")
)

display(
    amount_precision.agg(
        F.count("*").alias("input_rows"),
        F.count_if(
            F.col("wide_amount").isNull()
            | F.col("money_amount").isNull()
        ).alias("unparseable_or_out_of_range_amounts"),
        F.count_if(
            F.col("wide_amount")
            != F.col("money_amount").cast("decimal(38,18)")
        ).alias("amounts_changed_at_two_decimal_places")
    )
)